## 1. Utilize Machine Learning - based training recommendation for optimizing marathon performance

Traditional one-size-fits-all approaches increasingly demonstrate inadequate accommodation of substantial inter-individual differences in physiological responses, recovery capacity, and adaptation rates among runners.

Contemporary training prescription faces the fundamental challenge of reconciling standardized protocols with the inherent biological variability that characterizes human athletic performance, particularly in endurance sports where
training adaptations unfold over extended periods and involve complex physiological systems.

The integration of machine learning (ML) into training recommendation systems offers a promising avenue for addressing these challenges by leveraging data-driven insights to tailor training regimens to individual athletes.

## 2. Training Trajectory Similarity — Dynamic Time Warping
Compares the weekly [km, avg HR] fingerprint of the current Amsterdam block against the same first N weeks of each completed marathon to find the closest match.

In [ ]:
# Requires: pip install fastdtw

import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from scipy.spatial.distance import euclidean
from fastdtw import fastdtw
import os
from dotenv import load_dotenv
load_dotenv()

host     = os.getenv("SUPABASE_HOST")
user_db  = os.getenv("SUPABASE_USER")
password = os.getenv("SUPABASE_PASSWORD")
engine   = create_engine(f"postgresql+psycopg2://{user_db}:{password}@{host}:6543/postgres")

df_ml = pd.read_sql("""
    SELECT a.start_date, a.activity_type, a.distance, a.avg_heartrate, a.activity_name
    FROM activities a
    JOIN athletes ath ON a.athlete_id = ath.id
    ORDER BY a.start_date ASC
""", engine)
df_ml["start_date"] = pd.to_datetime(df_ml["start_date"])
runs_ml = df_ml[df_ml["activity_type"] == "Run"].dropna(subset=["distance"]).copy()

EXCLUDE = r"interval|sprint|tempo|fartlek|race|hill repeat|track workout"

def weekly_features(runs, start, end=None, n_weeks=None):
    """Returns array of shape (weeks, 2): [weekly_km, avg_hr_easy] per week."""
    mask = runs["start_date"] >= pd.Timestamp(start)
    if end:
        mask &= runs["start_date"] < pd.Timestamp(end)
    b = runs[mask].copy()
    if len(b) == 0:
        return np.empty((0, 2))
    b["week"] = (b["start_date"] - pd.Timestamp(start)).dt.days // 7
    km_w  = b.groupby("week")["distance"].sum()
    easy  = b[~b["activity_name"].str.contains(EXCLUDE, case=False, na=False)]
    hr_w  = easy.dropna(subset=["avg_heartrate"]).groupby("week")["avg_heartrate"].mean()
    max_w = int(km_w.index.max()) + 1
    if n_weeks:
        max_w = min(max_w, n_weeks)
    return np.array([[km_w.get(w, 0.0), hr_w.get(w, np.nan)] for w in range(max_w)], dtype=float)

# Extract features for each block
feat_current = weekly_features(runs_ml, "2026-06-15")
n = len(feat_current)
feat_1 = weekly_features(runs_ml, "2024-06-01", "2024-10-13", n_weeks=n)
feat_2 = weekly_features(runs_ml, "2025-10-20", "2026-05-10",  n_weeks=n)

# Normalize features using global mean/std so km and HR contribute equally
all_data = np.vstack([feat_1, feat_2, feat_current])
for col in range(2):
    mu  = np.nanmean(all_data[:, col])
    sig = np.nanstd(all_data[:, col]) or 1.0
    for f in [feat_1, feat_2, feat_current]:
        f[np.isnan(f[:, col]), col] = mu   # fill NaN HR with global mean
        f[:, col] = (f[:, col] - mu) / sig

# DTW similarity
dist_1, _ = fastdtw(feat_current, feat_1, dist=euclidean)
dist_2, _ = fastdtw(feat_current, feat_2, dist=euclidean)

closer = "Copenhagen 2026" if dist_2 < dist_1 else "Eindhoven 2024"
other  = "Eindhoven 2024"  if dist_2 < dist_1 else "Copenhagen 2026"

print(f"Weeks of Amsterdam data compared: {n}")
print(f"DTW distance — Eindhoven 2024:  {dist_1:.2f}")
print(f"DTW distance — Copenhagen 2026: {dist_2:.2f}")
print(f"\nAmsterdam is tracking closest to: {closer}")
print(f"({min(dist_1,dist_2):.2f} vs {max(dist_1,dist_2):.2f} for {other})")

## 3. Acute:Chronic Workload Ratio (ACWR) — Injury Risk Monitor

The ACWR compares the **acute** workload (last 7 days) against the **chronic** workload (rolling 28-day average weekly load).
A ratio between **0.8 – 1.3** is considered the "sweet spot" for adaptation without elevated injury risk.
Above **1.3** flags a spike; above **1.5** is high-risk territory.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker

# ── Build daily load series for Amsterdam block ──────────────────────────────
BLOCK_START = pd.Timestamp("2026-06-15")
amsterdam = runs_ml[runs_ml["start_date"] >= BLOCK_START].copy()
amsterdam["date"] = amsterdam["start_date"].dt.normalize()

daily = amsterdam.groupby("date")["distance"].sum().rename("km")
# fill gaps so rolling windows are contiguous
idx_full = pd.date_range(BLOCK_START, pd.Timestamp.today().normalize(), freq="D")
daily = daily.reindex(idx_full, fill_value=0.0)

acute   = daily.rolling(7,  min_periods=1).sum()          # 7-day rolling total
chronic = daily.rolling(28, min_periods=7).sum() / 4      # 28-day / 4 = avg weekly
acwr    = (acute / chronic).replace([np.inf, -np.inf], np.nan)

current_acwr    = acwr.dropna().iloc[-1] if acwr.dropna().size else np.nan
current_acute   = acute.iloc[-1]
current_chronic = chronic.iloc[-1]

# ── Plot ─────────────────────────────────────────────────────────────────────
BG      = "#1A1A2E"
CREAM   = "#C8B98A"
ACUTE_C = "#5B8DB8"
RISK_G  = "#2ECC71"   # green  0.8 – 1.3
RISK_O  = "#E67E22"   # orange 1.3 – 1.5
RISK_R  = "#E74C3C"   # red    > 1.5

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(14, 8), facecolor=BG,
    gridspec_kw={"height_ratios": [1.4, 1], "hspace": 0.08}
)
for ax in (ax_top, ax_bot):
    ax.set_facecolor(BG)
    ax.tick_params(colors="white", labelsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor("#444466")

dates = daily.index

# — top: daily bars + acute load line —
ax_top.bar(dates, daily.values, color=ACUTE_C, alpha=0.45, width=0.9, label="Daily km")
ax_top.plot(dates, acute.values, color=CREAM, linewidth=1.8, label="7-day rolling load")
ax_top.set_xlim(dates[0], dates[-1])
ax_top.set_ylabel("km", color="white", fontsize=10)
ax_top.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax_top.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
plt.setp(ax_top.get_xticklabels(), visible=False)
ax_top.legend(facecolor="#111122", edgecolor="#444466", labelcolor="white",
              fontsize=9, loc="upper left")
ax_top.set_title("ACWR — Amsterdam Marathon Block (from 15 Jun 2026)",
                 color="white", fontsize=13, pad=12)

# — bottom: ACWR line + coloured risk bands —
ax_bot.axhspan(0.8,  1.3,  color=RISK_G, alpha=0.12, zorder=0)
ax_bot.axhspan(1.3,  1.5,  color=RISK_O, alpha=0.18, zorder=0)
ax_bot.axhspan(1.5,  3.0,  color=RISK_R, alpha=0.18, zorder=0)
ax_bot.axhspan(0.0,  0.8,  color=RISK_O, alpha=0.10, zorder=0)  # undertraining zone

ax_bot.axhline(0.8,  color=RISK_G, linewidth=0.7, linestyle="--", alpha=0.6)
ax_bot.axhline(1.3,  color=RISK_O, linewidth=0.7, linestyle="--", alpha=0.6)
ax_bot.axhline(1.5,  color=RISK_R, linewidth=0.7, linestyle="--", alpha=0.6)

# zone labels (right-side)
for y, label, col in [(1.05, "Sweet spot", RISK_G), (1.4, "Caution", RISK_O),
                       (1.6, "High risk", RISK_R), (0.45, "Under-trained", RISK_O)]:
    ax_bot.text(dates[-1], y, f" {label}", color=col, fontsize=8,
                va="center", ha="left", clip_on=False)

# colour the ACWR line by zone
acwr_vals = acwr.values
for i in range(1, len(dates)):
    v = acwr_vals[i]
    if np.isnan(v):
        continue
    c = RISK_G if 0.8 <= v <= 1.3 else (RISK_O if v <= 1.5 else RISK_R)
    ax_bot.plot(dates[i-1:i+1], acwr_vals[i-1:i+1], color=c, linewidth=2.2)

ax_bot.set_xlim(dates[0], dates[-1])
ax_bot.set_ylim(0, max(acwr.dropna().max() * 1.15, 1.8))
ax_bot.set_ylabel("ACWR", color="white", fontsize=10)
ax_bot.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax_bot.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
plt.setp(ax_bot.get_xticklabels(), rotation=30, ha="right", color="white")

# current ACWR annotation
if not np.isnan(current_acwr):
    ax_bot.annotate(
        f"Now: {current_acwr:.2f}",
        xy=(dates[-1], current_acwr),
        xytext=(-60, 20), textcoords="offset points",
        color="white", fontsize=10, fontweight="bold",
        arrowprops=dict(arrowstyle="->", color="white", lw=1.2),
    )

plt.tight_layout()
plt.savefig(f"{os.getenv('FIGURES_DIR')}/acwr_amsterdam.png", dpi=150,
            bbox_inches="tight", facecolor=BG)
plt.show()

# ── Summary ──────────────────────────────────────────────────────────────────
zone = ("sweet spot ✓" if 0.8 <= current_acwr <= 1.3
        else ("caution — slight spike" if current_acwr <= 1.5
              else "HIGH RISK — back off"))
print(f"Acute load  (last 7 days):     {current_acute:.1f} km")
print(f"Chronic load (avg weekly/28d): {current_chronic:.1f} km")
print(f"Current ACWR:                  {current_acwr:.3f}  →  {zone}")